# GraphKV real vLLM + LMCache experiment


Runs the exact-chunk real-cache benchmark on one NVIDIA GPU. On Kaggle's two-T4 runtime, GPU 0 alone is exposed to the benchmark. Candidate population is real vLLM inference and its measured cost is included in end-to-end latency.


In [ ]:
from pathlib import Path
import os, subprocess, sys

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
VLLM = ROOT / 'vllm_lmcache'
assert (VLLM / 'run_repaired.py').exists(), VLLM
os.chdir(VLLM)
print('Working directory:', Path.cwd())


## Install
Run this once in a fresh Kaggle or Colab GPU runtime, then restart the Python runtime before continuing so vLLM, LMCache, Torch, and CUDA extensions are loaded consistently.


In [ ]:
subprocess.run(['bash', 'scripts/install_notebook.sh'], check=True)


## Configuration and preflight


In [ ]:
MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'
GPU = '0'
SMOKE_OUTPUT = '/kaggle/working/graphkv_hotpot_smoke'
FULL_OUTPUT = '/kaggle/working/hotpot_qwen15b_full_confirm'
# For Colab, change the two paths above to /content/outputs/...


In [ ]:
subprocess.run([sys.executable, 'run_repaired.py', '--dataset', 'hotpot',
    '--model', MODEL, '--gpu', GPU, '--preflight-only',
    '--output-dir', SMOKE_OUTPUT], check=True)


## Mandatory isolated cache controls


In [ ]:
subprocess.run([sys.executable, 'run_repaired.py', '--dataset', 'hotpot',
    '--model', MODEL, '--gpu', GPU, '--events', '5', '--train-events', '80',
    '--dev-events', '40', '--max-documents', '30',
    '--document-chunk-tokens', '384', '--document-chunk-overlap-tokens', '64',
    '--l1-size-gb', '0.5', '--k-values', '5', '--repetitions', '1',
    '--sanity-only', '--output-dir', SMOKE_OUTPUT], check=True)


## Five-event pipeline smoke


In [ ]:
subprocess.run([sys.executable, 'run_repaired.py', '--dataset', 'hotpot',
    '--model', MODEL, '--gpu', GPU, '--events', '5', '--train-events', '80',
    '--dev-events', '40', '--max-documents', '30',
    '--document-chunk-tokens', '384', '--document-chunk-overlap-tokens', '64',
    '--l1-size-gb', '0.5', '--k-values', '5', '--repetitions', '1',
    '--policies', 'no_prefetch,cosine,graph_fixed,adaptive_offline_global,adaptive_online_warm',
    '--output-dir', SMOKE_OUTPUT], check=True)


## Completed paper configuration
This is the exact scientific configuration recorded by the completed Qwen2.5-1.5B run: 600 test events, K={6,10,16}, two randomized repetitions, seed 43, and 30 isolated policy arms. It can take many hours on a T4.


In [ ]:
full_cmd = [sys.executable, 'run_repaired.py', '--dataset', 'hotpot',
    '--model', MODEL, '--gpu', GPU, '--events', '600', '--train-events', '600',
    '--dev-events', '600', '--max-documents', '100',
    '--max-chars-per-document', '16000', '--document-chunk-tokens', '384',
    '--document-chunk-overlap-tokens', '64', '--top-m', '20',
    '--l1-size-gb', '0.5', '--gpu-memory-utilization', '0.82',
    '--max-model-len', '4096', '--block-size', '16', '--chunk-size', '16',
    '--k-values', '6,10,16', '--repetitions', '2', '--seed', '43',
    '--policies', 'no_prefetch,cosine,graph_fixed,adaptive_offline_global,adaptive_online_warm',
    '--output-dir', FULL_OUTPUT]
subprocess.run(full_cmd, check=True)


Rerun the identical command after interruption. Completed arms are reused; an incomplete arm restarts in fresh processes.


## Validate, analyze, and package results


In [ ]:
subprocess.run([sys.executable, 'validate_repaired.py', FULL_OUTPUT], check=True)
subprocess.run([sys.executable, 'analyze_repaired.py', FULL_OUTPUT], check=True)
subprocess.run(['bash', 'scripts/package_results.sh', FULL_OUTPUT], check=True)
